In [1]:
needed_tables = ['auth_account_purview',
'auth_account_role',
'auth_menu_purview',
'auth_purview',
'auth_role',
'auth_role_purview',
'auth_tenant_privileges',
'auth_user',
'auth_user_auth',
'auth_user_base',
'auth_user_data_permission',
'auth_user_properties_ext',
'auth_user_role',
'auth_wx_config',
'biz_log_record',
'biz_log_template',
'business_data_permission',
'business_information',
'download_center_record_mapping',
'download_center_type_config',
'file_download_record',
'item_sale_limit_config',
'market',
'market_area_item',
'market_area_item_mapping',
'market_area_item_store_price_mapping',
'market_classification',
'market_combine_item_mapping',
'market_detail',
'market_item',
'market_item_availability_change_record',
'market_item_classification',
'market_item_detail',
'market_item_label',
'market_item_onsale_strategy_mapping',
'market_item_order_summary',
'market_item_price',
'market_item_price_log',
'market_item_price_strategy',
'market_item_price_strategy_mapping',
'market_item_unfair_price_strategy',
'market_item_unit',
'merchant',
'merchant_address',
'merchant_contact',
'merchant_delivery_fee_rule',
'merchant_order_quantity_rule',
'merchant_store',
'merchant_store_account',
'merchant_store_account_operator_log',
'merchant_store_change_log',
'merchant_store_config',
'merchant_store_ext',
'merchant_store_group',
'merchant_store_group_mapping',
'merchant_store_properties_ext',
'msg_scene',
'msg_scene_tenant_mapping',
'order',
'order_address',
'order_after_sale',
'order_after_sale_rule',
'order_combine_snapshot',
'order_delivery_detail',
'order_delivery_fee_snapshot',
'order_delivery_info',
'order_item',
'order_item_extra',
'order_item_fee_transaction',
'order_item_snapshot',
'order_snapshot',
'regional_organization',
'sms_scene',
'tenant',
'tenant_account',
'tenant_account_bussiness_msg_config',
'tenant_account_receive_msg_switch',
'tenant_account_supplier_mapping',
'tenant_agreement',
'tenant_common_config',
'tenant_company',
'tenant_company_account',
'tenant_data_value',
'tenant_ext_sys_config',
'tenant_function_set',
'tenant_fund_account',
'tenant_fund_account_config',
'tenant_privileges_config',
'tenant_store_common_config',
'tenant_switch',
'tenant_template_init',
'trolley',]

In [2]:
import pymysql

tables = []
def get_all_table_ddls(host, port, user, password, database, charset='utf8'):
    global tables
    """
    连接到MySQL数据库并获取指定数据库下所有表的DDL。

    Args:
        host (str): 数据库主机名或IP地址。
        port (int): 数据库端口号。
        user (str): 数据库用户名。
        password (str): 数据库密码。
        database (str): 要获取DDL的数据库名称。
        charset (str): 连接字符集，默认为'utf8'。

    Returns:
        dict: 一个字典，键是表名，值是对应的CREATE TABLE语句。
              如果连接失败或没有表，则返回空字典。
    """
    conn = None
    cursor = None
    table_ddls = {}
    try:
        # 建立数据库连接
        conn = pymysql.connect(
            host=host,
            port=port,
            user=user,
            password=password,
            database=database,
            charset=charset,
            # useSSL=false 对应 ssl=False
            ssl=False
            # serverTimezone=GMT%2B8 和 characterEncoding=utf-8 已经在 charset 中处理
        )
        cursor = conn.cursor()

        # 获取所有表名
        cursor.execute(f"SHOW TABLES IN `{database}`")
        tables = [row[0] for row in cursor.fetchall()]

        # 遍历表名，获取每个表的DDL
        for table_name in tables:
            # SHOW CREATE TABLE 返回两列：表名和 CREATE TABLE 语句
            cursor.execute(f"SHOW CREATE TABLE `{table_name}`")
            result = cursor.fetchone()
            if result and len(result) > 1:
                table_ddls[table_name] = result[1] # 第二列是 CREATE TABLE 语句

    except pymysql.Error as e:
        print(f"数据库连接或查询错误: {e}")
        # 可以选择重新抛出异常或返回空字典
        # raise e
    finally:
        # 确保关闭游标和连接
        if cursor:
            cursor.close()
        if conn:
            conn.close()

    return table_ddls

# 示例用法 (使用注释中的连接信息)
db_host = "mysql-8.summerfarm.net"
db_port = 3307
db_user = "dev"
db_password = "xianmu619"
db_name = "wurthdb"

all_ddls = get_all_table_ddls(db_host, db_port, db_user, db_password, db_name)



from openai import OpenAI
import os

# 初始化OpenAI客户端
client = OpenAI(
    base_url="https://test-one-api.summerfarm.top/v1",
    api_key=os.getenv("XM_FAST_GPT_API_KEY")
)


In [5]:
def optimize_ddl(ddl):
    prompt = f"""请帮我优化以下MySQL DDL语句：
    1. 如果表中有auto_increment字段，将初始ID设置为10000
    2. 将所有表的utf8mb3字符集改为utf8mb4
    3. 将所有的`CREATE TABLE`语句改为`CREATE TABLE IF NOT EXISTS`
    3. 如果语句后面没有`;`，请加上`;`
    3. 保持其他部分不变.
    4. 仅返回优化后的DDL语句，不要返回任何解释或说明，不要添加任何额外的内容。
    
    原始DDL：
    {ddl}
    """
    
    response = client.chat.completions.create(
        model="deepseek-v3-250324",
        messages=[
            {"role": "system", "content": "你是一个专业的MySQL数据库管理员"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

from concurrent.futures import ThreadPoolExecutor

def process_table(table_ddl):
    global needed_tables
    table, ddl = table_ddl
    if table not in needed_tables:
        return
    optimized_ddl = optimize_ddl(ddl)
    print(f"--- Optimized DDL for table: {table} ---")
    print(f"{optimized_ddl}\n")
    print("-" * 40)

if all_ddls:
    with ThreadPoolExecutor(max_workers=10) as executor:
        # 将字典项转换为可迭代的元组列表
        executor.map(process_table, all_ddls.items())
else:
    print("未能获取到任何表的DDL。")
    
    
    
    

--- Optimized DDL for table: auth_tenant_privileges ---
CREATE TABLE IF NOT EXISTS `auth_tenant_privileges` (
  `id` bigint unsigned NOT NULL AUTO_INCREMENT COMMENT 'primary key',
  `create_time` datetime NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'create time',
  `update_time` datetime DEFAULT NULL ON UPDATE CURRENT_TIMESTAMP COMMENT 'update time',
  `tenant_id` bigint DEFAULT NULL COMMENT '租户id',
  `menu_id` bigint DEFAULT NULL COMMENT '权益id',
  `expire_time` datetime DEFAULT NULL COMMENT '过期时间',
  PRIMARY KEY (`id`),
  UNIQUE KEY `uk_tenant_id_menu_id` (`tenant_id`,`menu_id`)
) ENGINE=InnoDB AUTO_INCREMENT=10000 DEFAULT CHARSET=utf8mb4 COMMENT='租户权益表';

----------------------------------------
--- Optimized DDL for table: auth_account_purview ---
CREATE TABLE IF NOT EXISTS `auth_account_purview` (
  `id` int NOT NULL AUTO_INCREMENT COMMENT 'Id',
  `role_id` int NOT NULL COMMENT '角色',
  `purview_id` int NOT NULL COMMENT '权限',
  `tenant_id` bigint DEFAULT NULL COMMENT '租户id cosfo tena

In [ ]:
all_ddls['order']

In [ ]:
needed_tables = ['auth_account_purview',
'auth_account_role',
'auth_menu_purview',
'auth_purview',
'auth_role',
'auth_role_purview',
'auth_tenant_privileges',
'auth_user',
'auth_user_auth',
'auth_user_base',
'auth_user_data_permission',
'auth_user_properties_ext',
'auth_user_role',
'auth_wx_config',
'biz_log_record',
'biz_log_template',
'business_data_permission',
'business_information',
'download_center_record_mapping',
'download_center_type_config',
'file_download_record',
'item_sale_limit_config',
'market',
'market_area_item',
'market_area_item_mapping',
'market_area_item_store_price_mapping',
'market_classification',
'market_combine_item_mapping',
'market_detail',
'market_item',
'market_item_availability_change_record',
'market_item_classification',
'market_item_detail',
'market_item_label',
'market_item_onsale_strategy_mapping',
'market_item_order_summary',
'market_item_price',
'market_item_price_log',
'market_item_price_strategy',
'market_item_price_strategy_mapping',
'market_item_unfair_price_strategy',
'market_item_unit',
'merchant',
'merchant_address',
'merchant_contact',
'merchant_delivery_fee_rule',
'merchant_order_quantity_rule',
'merchant_store',
'merchant_store_account',
'merchant_store_account_operator_log',
'merchant_store_change_log',
'merchant_store_config',
'merchant_store_ext',
'merchant_store_group',
'merchant_store_group_mapping',
'merchant_store_properties_ext',
'msg_scene',
'msg_scene_tenant_mapping',
'order',
'order_address',
'order_after_sale',
'order_after_sale_rule',
'order_combine_snapshot',
'order_delivery_detail',
'order_delivery_fee_snapshot',
'order_delivery_info',
'order_item',
'order_item_extra',
'order_item_fee_transaction',
'order_item_snapshot',
'order_snapshot',
'regional_organization',
'sms_scene',
'tenant',
'tenant_account',
'tenant_account_bussiness_msg_config',
'tenant_account_receive_msg_switch',
'tenant_account_supplier_mapping',
'tenant_agreement',
'tenant_common_config',
'tenant_company',
'tenant_company_account',
'tenant_data_value',
'tenant_ext_sys_config',
'tenant_function_set',
'tenant_fund_account',
'tenant_fund_account_config',
'tenant_privileges_config',
'tenant_store_common_config',
'tenant_switch',
'tenant_template_init',
'trolley',]

In [ ]:
wurth_tables = [
    "auth_menu_purview",
    "auth_role",
    "auth_role_purview",
    "auth_tenant_privileges",
    "auth_user",
    "auth_user_auth",
    "auth_user_base",
    "auth_user_data_permission",
    "auth_user_properties_ext",
    "auth_user_role",
    "auth_wechat_relation",
    "auth_wx_config",
    "business_data_permission",
]

wurth_tables: set = set(wurth_tables)
# 示例用法 (使用注释中的连接信息)
db_host = "mysql-8.summerfarm.net"
db_port = 3307
db_user = "dev"
db_password = "xianmu619"
db_name = "wurthdb"

try:
    # 建立数据库连接
    conn = pymysql.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database=db_name,
        # useSSL=false 对应 ssl=False
        ssl=False,
        # serverTimezone=GMT%2B8 和 characterEncoding=utf-8 已经在 charset 中处理
    )
    cursor = conn.cursor()

    # 获取所有表名
    for table_name, ddl in all_ddls.items():
        if table_name not in wurth_tables:
            print(f"skip table: {table_name}")
            continue

        # 使用正则表达式替换 AUTO_INCREMENT 的起始值
        import re

        ddl = re.sub(r"AUTO_INCREMENT=\d+", "AUTO_INCREMENT=10000", ddl)
        ddl = ddl.replace('CREATE TABLE', 'CREATE TABLE IF NOT EXISTS')
        print(f"create table: {table_name} with ddl: {ddl}")

        cursor.execute(f"{ddl};")
        result = cursor.fetchone()
        if result and len(result) > 1:
            print(f"--- DDL for table: {table_name} ---")
            print(result[1])
            print("-" * 20)

except pymysql.Error as e:
    print(f"数据库连接或查询错误: {e}")
    # 可以选择重新抛出异常或返回空字典
    # raise e
finally:
    # 确保关闭游标和连接
    if cursor:
        cursor.close()
    if conn:
        conn.close()